In [ ]:
import os
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from joblib import Parallel, delayed
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
np.set_printoptions(precision=6, suppress=True)

SEED = 42
N_SPLITS = 10
MAX_SEQ_EPOCHS = 12
SEQ_PATIENCE = 3
TABULAR_PARALLEL_JOBS = max(1, min(4, os.cpu_count() or 4))
MODEL_THREAD_COUNT = 1

DATA_ROOT = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
if not DATA_ROOT.exists():
    DATA_ROOT = Path.cwd() / "competitions" / "rogii-wellbore-geology-prediction"
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
ARTIFACT_DIR = Path.cwd() / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)

DEBUG_MODE = True 


def seed_everything(seed: int = 42) -> None:
    """Seed Python, NumPy, and PyTorch for deterministic runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"

print(f"Seed set to: {SEED}")
print(f"Device: {DEVICE}")
print(f"Train dir exists: {TRAIN_DIR.exists()} | {TRAIN_DIR}")
print(f"Test dir exists: {TEST_DIR.exists()} | {TEST_DIR}")
print(f"Artifacts dir: {ARTIFACT_DIR}")
print(f"Library check -> torch {torch.__version__}, pandas {pd.__version__}, numpy {np.__version__}")

In [ ]:
from typing import Dict, List, Tuple


def extract_wellname(path: Path) -> str:
    """Extract WELLNAME from a file name like WELL__horizontal_well.csv."""
    name = path.name
    return name.split("__", 1)[0] if "__" in name else path.stem


def read_csv_safe(path: Path) -> pd.DataFrame:
    """Read a CSV and attach the original row order for later alignment."""
    df = pd.read_csv(path).copy()
    df["row_idx"] = np.arange(len(df), dtype=np.int64)
    return df


def _nan_run_profile(values: np.ndarray) -> Tuple[List[int], List[int]]:
    starts, lengths = [], []
    in_run = False
    run_start = None
    for idx, is_nan in enumerate(np.isnan(values)):
        if is_nan and not in_run:
            in_run = True
            run_start = idx
        elif not is_nan and in_run:
            in_run = False
            starts.append(int(run_start))
            lengths.append(int(idx - run_start))
    if in_run and run_start is not None:
        starts.append(int(run_start))
        lengths.append(int(len(values) - run_start))
    return starts, lengths


def audit_tvt_boundary(df: pd.DataFrame, source_col: str | None = None, window: int = 5) -> dict:
    """Profile the TVT/TVT_input missingness topology and extract boundary features."""
    if source_col is None:
        if "TVT_input" in df.columns:
            source_col = "TVT_input"
        elif "TVT" in df.columns:
            source_col = "TVT"
        else:
            source_col = None

    if source_col is None:
        values = np.full(len(df), np.nan, dtype=float)
    else:
        values = pd.to_numeric(df[source_col], errors="coerce").to_numpy(dtype=float, copy=True)

    valid = np.isfinite(values)
    if valid.any():
        valid_idx = np.flatnonzero(valid)
        last_valid_idx = int(valid_idx[-1])
        transitions = np.flatnonzero(valid[:-1] & ~valid[1:]) + 1 if len(values) > 1 else np.array([], dtype=int)
        eval_start_idx = int(transitions[0]) if transitions.size else int(np.flatnonzero(~valid)[0]) if (~valid).any() else len(values)
        tail_start = max(0, last_valid_idx - window + 1)
        tail_vals = values[tail_start:last_valid_idx + 1]
        x = np.arange(len(tail_vals), dtype=float)
        local_gradient = float(np.polyfit(x, tail_vals, deg=1)[0]) if len(tail_vals) >= 2 else 0.0
        if len(tail_vals) >= 3:
            curvature = float(np.gradient(np.gradient(tail_vals))[-1])
        else:
            curvature = 0.0
        last_valid_value = float(values[last_valid_idx])
        nan_starts, nan_lengths = _nan_run_profile(values)
        terminal_nan_run = bool(eval_start_idx < len(values) and np.all(np.isnan(values[eval_start_idx:])))
    else:
        last_valid_idx = -1
        eval_start_idx = len(values)
        local_gradient = 0.0
        curvature = 0.0
        last_valid_value = np.nan
        nan_starts, nan_lengths = [], []
        terminal_nan_run = False

    missing_fraction = float(np.isnan(values).mean()) if len(values) else 0.0
    valid_prefix_len = int(eval_start_idx if eval_start_idx <= len(values) else len(values))

    return {
        "tvt_source_col": source_col,
        "tvt_eval_start_idx": int(eval_start_idx),
        "tvt_last_valid_idx": int(last_valid_idx),
        "tvt_last_valid_value": last_valid_value,
        "tvt_local_gradient": float(local_gradient),
        "tvt_curvature": float(curvature),
        "tvt_missing_fraction": missing_fraction,
        "tvt_valid_prefix_len": valid_prefix_len,
        "tvt_nan_run_starts": nan_starts,
        "tvt_nan_run_lengths": nan_lengths,
        "tvt_terminal_nan_run": terminal_nan_run,
    }


def build_boundary_features(df: pd.DataFrame, boundary: dict) -> pd.DataFrame:
    """Broadcast boundary audit features to every row in a well dataframe."""
    out = df.copy()
    for key, value in boundary.items():
        if isinstance(value, (list, tuple, np.ndarray)):
            continue
        out[key] = value
    out["tvt_rows_from_boundary"] = out["row_idx"] - out["tvt_eval_start_idx"]
    out["tvt_is_eval_zone"] = (out["row_idx"] >= out["tvt_eval_start_idx"]).astype(np.int8)
    out["tvt_has_boundary"] = int(out["tvt_eval_start_idx"].iloc[0] < len(out))
    return out


def load_well_directory(base_dir: Path) -> List[dict]:
    """Load horizontal/typewell CSV pairs from a directory, grouped by WELLNAME."""
    horizontal_paths = sorted(base_dir.glob("*__horizontal_well.csv"))
    typewell_paths = sorted(base_dir.glob("*__typewell.csv"))
    horizontal_map = {extract_wellname(p): p for p in horizontal_paths}
    typewell_map = {extract_wellname(p): p for p in typewell_paths}
    well_names = sorted(set(horizontal_map) & set(typewell_map))

    bundles = []
    for well_name in well_names:
        h_path = horizontal_map[well_name]
        t_path = typewell_map[well_name]
        h_df = read_csv_safe(h_path)
        t_df = read_csv_safe(t_path)
        h_df["WELLNAME"] = well_name
        t_df["WELLNAME"] = well_name
        h_df["Horizontal_Well_GR"] = pd.to_numeric(h_df.get("GR"), errors="coerce")
        t_df["Typewell_GR"] = pd.to_numeric(t_df.get("GR"), errors="coerce")
        t_df["Typewell_TVT"] = pd.to_numeric(t_df.get("TVT"), errors="coerce") if "TVT" in t_df.columns else np.nan
        boundary = audit_tvt_boundary(h_df)
        h_df = build_boundary_features(h_df, boundary)
        bundles.append(
            {
                "WELLNAME": well_name,
                "horizontal": h_df,
                "typewell": t_df,
                "boundary": boundary,
                "horizontal_path": h_path,
                "typewell_path": t_path,
            }
        )
    return bundles


train_wells = load_well_directory(TRAIN_DIR)
test_wells = load_well_directory(TEST_DIR)
train_well_map = {w["WELLNAME"]: w for w in train_wells}
test_well_map = {w["WELLNAME"]: w for w in test_wells}

print(f"Training wells loaded: {len(train_wells)}")
print(f"Test wells loaded: {len(test_wells)}")
if train_wells:
    sample = train_wells[0]
    print(f"Sample train horizontal shape: {sample['horizontal'].shape}")
    print(f"Sample train typewell shape: {sample['typewell'].shape}")
    print(f"Boundary audit preview: {sample['boundary']}")
    print(f"Train horizontal columns: {list(sample['horizontal'].columns[:20])}")

In [ ]:
SURFACE_COLUMNS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
LEAKAGE_COLS = ["tvt_missing_fraction", "tvt_valid_prefix_len"]
BASE_IGNORE_COLUMNS = {
    "TVT",
    "TVT_input",
    "target_tvt",
    "dtvt",
    "WELLNAME",
    "row_idx",
    "horizontal_path",
    "typewell_path",
}


def build_surface_lookup_from_typewell(typewell_df: pd.DataFrame) -> dict:
    """Extract formation-top TVT depths from geology transitions in the typewell log."""
    tw = typewell_df.copy()
    tw["TVT"] = pd.to_numeric(tw.get("TVT"), errors="coerce")
    tw["Geology"] = tw.get("Geology", pd.Series(index=tw.index, dtype=object)).astype(str).str.strip().str.upper()
    tw = tw.dropna(subset=["TVT", "Geology"]).sort_values("TVT").reset_index(drop=True)
    if tw.empty:
        return {surface: np.nan for surface in SURFACE_COLUMNS}
    segment_starts = tw["Geology"].ne(tw["Geology"].shift())
    tops = tw.loc[segment_starts, ["Geology", "TVT"]]
    label_to_tvt = {row.Geology: float(row.TVT) for row in tops.itertuples(index=False)}
    return {surface: float(label_to_tvt.get(surface, np.nan)) for surface in SURFACE_COLUMNS}


def fit_surface_fallback_models(train_wells: List[dict]) -> dict:
    """Train one auxiliary regressor per surface using spatial coordinates only."""
    train_frames = []
    for bundle in train_wells:
        h = bundle["horizontal"][ ["MD", "X", "Y", "Z"] + SURFACE_COLUMNS ].copy()
        h["WELLNAME"] = bundle["WELLNAME"]
        train_frames.append(h)
    surface_train = pd.concat(train_frames, ignore_index=True)
    surface_train = surface_train.apply(pd.to_numeric, errors="ignore")

    feature_cols = ["MD", "X", "Y", "Z"]
    models = {}
    medians = {}
    for surface in SURFACE_COLUMNS:
        df = surface_train[feature_cols + [surface]].copy()
        df = df.dropna(subset=[surface])
        if df.empty:
            models[surface] = None
            medians[surface] = np.nan
            continue
        X = df[feature_cols].copy()
        for col in feature_cols:
            X[col] = pd.to_numeric(X[col], errors="coerce")
            X[col] = X[col].interpolate(limit_direction="both").bfill().ffill()
        y = pd.to_numeric(df[surface], errors="coerce").to_numpy(dtype=float)
        model = lgb.LGBMRegressor(
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=64,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=SEED,
            n_jobs=MODEL_THREAD_COUNT,
            objective="regression",
            verbosity=-1,
        )
        model.fit(X.to_numpy(dtype=float), y)
        models[surface] = model
        medians[surface] = float(np.nanmedian(y))
    return {"models": models, "medians": medians, "feature_cols": feature_cols}


def predict_surface_fallback(surface_model_bundle: dict, horizontal_df: pd.DataFrame, surface: str) -> float:
    """Predict a missing surface depth from spatial geometry and reduce it to a single well-level marker."""
    model = surface_model_bundle["models"].get(surface)
    fallback_value = surface_model_bundle["medians"].get(surface, np.nan)
    if model is None:
        return float(fallback_value)
    X = horizontal_df[["MD", "X", "Y", "Z"]].copy()
    for col in ["MD", "X", "Y", "Z"]:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X[col] = X[col].interpolate(limit_direction="both").bfill().ffill()
    pred = model.predict(X.to_numpy(dtype=float))
    pred = pred[np.isfinite(pred)]
    if len(pred) == 0:
        return float(fallback_value)
    return float(np.nanmedian(pred))


def attach_surface_depths(horizontal_df: pd.DataFrame, surface_lookup: dict) -> pd.DataFrame:
    """Broadcast surface depths across every row of a horizontal well."""
    out = horizontal_df.copy()
    for surface in SURFACE_COLUMNS:
        out[surface] = float(surface_lookup.get(surface, np.nan))
    return out


surface_fallback_bundle = fit_surface_fallback_models(train_wells)
print("Surface fallback models trained for:", list(surface_fallback_bundle["models"].keys()))


def build_hybrid_surface_lookup(bundle: dict) -> dict:
    """Prefer typewell-derived tops and fall back to a spatial predictor when needed."""
    lookup = build_surface_lookup_from_typewell(bundle["typewell"])
    for surface in SURFACE_COLUMNS:
        if not np.isfinite(lookup.get(surface, np.nan)):
            lookup[surface] = predict_surface_fallback(surface_fallback_bundle, bundle["horizontal"], surface)
    return lookup


# Rebuild the well maps with non-NaN geological surfaces for both train and test.
train_wells_hybrid = []
test_wells_hybrid = []
for bundle in train_wells:
    surface_lookup = build_hybrid_surface_lookup(bundle)
    updated = dict(bundle)
    updated["surface_lookup"] = surface_lookup
    updated["horizontal"] = attach_surface_depths(bundle["horizontal"], surface_lookup)
    train_wells_hybrid.append(updated)
for bundle in test_wells:
    surface_lookup = build_hybrid_surface_lookup(bundle)
    updated = dict(bundle)
    updated["surface_lookup"] = surface_lookup
    updated["horizontal"] = attach_surface_depths(bundle["horizontal"], surface_lookup)
    test_wells_hybrid.append(updated)

train_wells = train_wells_hybrid
test_wells = test_wells_hybrid
train_well_map = {w["WELLNAME"]: w for w in train_wells}
test_well_map = {w["WELLNAME"]: w for w in test_wells}

train_surface_nan_counts = {surface: int(pd.concat([w["horizontal"][surface] for w in train_wells]).isna().sum()) for surface in SURFACE_COLUMNS}
test_surface_nan_counts = {surface: int(pd.concat([w["horizontal"][surface] for w in test_wells]).isna().sum()) for surface in SURFACE_COLUMNS}
print("Train surface NaN counts after hybrid fill:", train_surface_nan_counts)
print("Test surface NaN counts after hybrid fill:", test_surface_nan_counts)
if train_wells:
    print(f"Sample train surface lookup: {train_wells[0]['surface_lookup']}")
if test_wells:
    print(f"Sample test surface lookup: {test_wells[0]['surface_lookup']}")


def engineer_tabular_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build tabular features with guaranteed surface coverage."""
    out = df.copy()
    for col in ["MD", "X", "Y", "Z", "GR", "Horizontal_Well_GR"] + SURFACE_COLUMNS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "Z" in out.columns:
        for surface in SURFACE_COLUMNS:
            out[f"abs_Z_minus_{surface}"] = (out["Z"] - out[surface]).abs()

    if "GR" in out.columns:
        out["GR_roll_mean_5"] = out["GR"].rolling(5, min_periods=1).mean()
        out["GR_roll_std_5"] = out["GR"].rolling(5, min_periods=1).std().fillna(0.0)
        out["GR_diff_1"] = out["GR"].diff().fillna(0.0)
    else:
        out["GR_roll_mean_5"] = np.nan
        out["GR_roll_std_5"] = np.nan
        out["GR_diff_1"] = np.nan

    if "MD" in out.columns:
        out["MD_step"] = out["MD"].diff().fillna(0.0)
    else:
        out["MD_step"] = np.nan

    if "tvt_eval_start_idx" in out.columns:
        out["tvt_rows_from_boundary"] = out["row_idx"] - out["tvt_eval_start_idx"]
        out["tvt_is_eval_zone"] = (out["row_idx"] >= out["tvt_eval_start_idx"]).astype(np.int8)
        out["tvt_boundary_distance"] = out["row_idx"] - out["tvt_eval_start_idx"]
    else:
        out["tvt_rows_from_boundary"] = np.nan
        out["tvt_is_eval_zone"] = 0
        out["tvt_boundary_distance"] = np.nan

    return out


train_feature_frames = [engineer_tabular_features(bundle["horizontal"]) for bundle in train_wells]
test_feature_frames = [engineer_tabular_features(bundle["horizontal"]) for bundle in test_wells]

train_tabular_df = pd.concat(train_feature_frames, ignore_index=True)
test_tabular_df = pd.concat(test_feature_frames, ignore_index=True)

if "TVT" in train_tabular_df.columns:
    train_tabular_df["target_tvt"] = pd.to_numeric(train_tabular_df["TVT"], errors="coerce")
else:
    train_tabular_df["target_tvt"] = np.nan

# Switch the learning target to relative thickness change to remove large absolute-TVT offsets.
train_tabular_df = train_tabular_df.sort_values(["WELLNAME", "row_idx"]).reset_index(drop=True)
train_tabular_df["dtvt"] = train_tabular_df.groupby("WELLNAME")["TVT"].diff().fillna(0.0)
if "TVT_input" in train_tabular_df.columns:
    train_tabular_df["dtvt_anchor"] = train_tabular_df.groupby("WELLNAME")["TVT_input"].transform(lambda s: pd.to_numeric(s, errors="coerce").ffill().iloc[0] if s.notna().any() else np.nan)

train_tabular_df = train_tabular_df[np.isfinite(train_tabular_df["dtvt"])].reset_index(drop=True)

feature_columns = []
for col in train_tabular_df.columns:
    if col in BASE_IGNORE_COLUMNS:
        continue
    if col in {"target_tvt", "dtvt", "dtvt_anchor"}:
        continue
    if pd.api.types.is_numeric_dtype(train_tabular_df[col]):
        feature_columns.append(col)

# Explicitly remove target leakage dimensions
final_modeling_features = [col for col in feature_columns if col not in LEAKAGE_COLS]
final_modeling_features = [col for col in final_modeling_features if col not in {"TVT", "TVT_input"}]
final_modeling_features = list(dict.fromkeys(final_modeling_features))

X_train_tabular = train_tabular_df[final_modeling_features].apply(pd.to_numeric, errors="coerce")
y_train = train_tabular_df["dtvt"].to_numpy(dtype=np.float32)
groups = train_tabular_df["WELLNAME"].to_numpy()
X_test_tabular = test_tabular_df[final_modeling_features].apply(pd.to_numeric, errors="coerce")

print(f"Train tabular dataframe shape: {train_tabular_df.shape}")
print(f"Test tabular dataframe shape: {test_tabular_df.shape}")
print(f"Feature matrix shape: {X_train_tabular.shape} | Test matrix shape: {X_test_tabular.shape}")
print(f"Unique train wells: {pd.Series(groups).nunique()} | Unique test wells: {test_tabular_df['WELLNAME'].nunique()}")
print(f"Final modeling features ({len(final_modeling_features)}): {final_modeling_features[:20]}")
print(f"Leakage cols removed: {[c for c in LEAKAGE_COLS if c not in final_modeling_features]}")
print(f"Target dtvt summary -> mean: {float(np.nanmean(y_train)):.6f}, std: {float(np.nanstd(y_train)):.6f}")



def _fit_fold_tabular(model_name: str, fold_idx: int, train_idx: np.ndarray, val_idx: np.ndarray, X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame):
    """Fit one fold of a tree regressor and return OOF plus test predictions."""
    X_tr = X.iloc[train_idx].to_numpy(dtype=np.float32)
    y_tr = y[train_idx]
    X_va = X.iloc[val_idx].to_numpy(dtype=np.float32)
    y_va = y[val_idx]
    X_te = X_test.to_numpy(dtype=np.float32)

    if model_name == "lgbm":
        model = lgb.LGBMRegressor(
            n_estimators=5000,
            learning_rate=0.02,
            num_leaves=128,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=SEED,
            n_jobs=MODEL_THREAD_COUNT,
            objective="regression",
            verbosity=-1,
        )
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
    elif model_name == "catboost":
        model = CatBoostRegressor(
            iterations=5000,
            learning_rate=0.03,
            depth=8,
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=SEED,
            od_type="Iter",
            od_wait=100,
            verbose=False,
            thread_count=MODEL_THREAD_COUNT,
            allow_writing_files=False,
            task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    elif model_name == "xgboost":
        model = XGBRegressor(
            n_estimators=5000,
            learning_rate=0.03,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            early_stopping_rounds=100,
            objective="reg:squarederror",
            random_state=SEED,
            n_jobs=MODEL_THREAD_COUNT,
            tree_method="hist",
            verbosity=0,
        )
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )
    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    val_pred = model.predict(X_va)
    test_pred = model.predict(X_te)
    fold_rmse = rmse(y_va, val_pred)
    return {
        "fold_idx": fold_idx,
        "model": model,
        "val_idx": val_idx,
        "val_pred": val_pred,
        "test_pred": test_pred,
        "fold_rmse": fold_rmse,
    }


def train_groupkfold_tabular(model_name: str, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, X_test: pd.DataFrame, n_splits: int = N_SPLITS):
    """Parallelized GroupKFold training loop for one tabular model family."""
    gkf = GroupKFold(n_splits=n_splits)
    fold_splits = list(gkf.split(X, y, groups))
    fold_results = Parallel(n_jobs=TABULAR_PARALLEL_JOBS, prefer="processes")(
        delayed(_fit_fold_tabular)(model_name, fold_idx, train_idx, val_idx, X, y, X_test)
        for fold_idx, (train_idx, val_idx) in enumerate(fold_splits)
    )
    oof = np.zeros(len(X), dtype=np.float32)
    test_pred = np.zeros(len(X_test), dtype=np.float32)
    models = []
    for res in sorted(fold_results, key=lambda d: d["fold_idx"]):
        oof[res["val_idx"]] = res["val_pred"]
        test_pred += res["test_pred"] / n_splits
        models.append(res["model"])
    score = rmse(y, oof)
    return {
        "model_name": model_name,
        "oof": oof,
        "test_pred": test_pred,
        "score": score,
        "models": models,
    }


tabular_results = {}
for family_name in ["lgbm", "catboost", "xgboost"]:
    result = train_groupkfold_tabular(family_name, X_train_tabular, y_train, groups, X_test_tabular, N_SPLITS)
    tabular_results[family_name] = result
    print(f"{family_name.upper()} OOF dtvt RMSE: {result['score']:.6f}")

train_tabular_df["oof_lgbm_dtvt"] = tabular_results["lgbm"]["oof"]
train_tabular_df["oof_catboost_dtvt"] = tabular_results["catboost"]["oof"]
train_tabular_df["oof_xgb_dtvt"] = tabular_results["xgboost"]["oof"]

print("Tabular OOF shapes:")
for family_name, result in tabular_results.items():
    print(f"  {family_name}: {result['oof'].shape} | test: {result['test_pred'].shape}")
print(f"Tabular OOF feature columns added: {[c for c in ['oof_lgbm_dtvt','oof_catboost_dtvt','oof_xgb_dtvt'] if c in train_tabular_df.columns]}")


In [ ]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # Safe across all scikit-learn versions
    mse = mean_squared_error(y_true, y_pred)
    return float(np.sqrt(mse))

def _fill_series_for_alignment(values: np.ndarray) -> np.ndarray:
    """Interpolate missing values before alignment calculations."""
    series = pd.Series(values.astype(float))
    series = series.interpolate(limit_direction="both").bfill().ffill()
    return series.to_numpy(dtype=np.float32)


def normalized_cross_correlation_lag(horizontal_gr: np.ndarray, vertical_gr: np.ndarray, max_lag: int | None = None) -> int:
    """Estimate a global lag using normalized cross-correlation over resampled GR sequences."""
    h = _fill_series_for_alignment(horizontal_gr)
    v = _fill_series_for_alignment(vertical_gr)
    common_len = int(min(max(len(h), len(v)), 512))
    x_common = np.linspace(0.0, 1.0, common_len)
    h_resampled = np.interp(x_common, np.linspace(0.0, 1.0, len(h)), (h - h.mean()) / (h.std() + 1e-6))
    v_resampled = np.interp(x_common, np.linspace(0.0, 1.0, len(v)), (v - v.mean()) / (v.std() + 1e-6))
    if max_lag is None:
        max_lag = max(1, min(common_len // 4, 75))
    best_lag = 0
    best_score = -np.inf
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            h_slice = h_resampled[-lag:]
            v_slice = v_resampled[: len(h_slice)]
        elif lag > 0:
            h_slice = h_resampled[: common_len - lag]
            v_slice = v_resampled[lag: lag + len(h_slice)]
        else:
            h_slice = h_resampled
            v_slice = v_resampled
        if len(h_slice) < 3:
            continue
        score = float(np.dot(h_slice, v_slice) / (np.linalg.norm(h_slice) * np.linalg.norm(v_slice) + 1e-6))
        if score > best_score:
            best_score = score
            best_lag = lag
    return int(best_lag)


def classical_alignment_predict(horizontal_df: pd.DataFrame, typewell_df: pd.DataFrame) -> np.ndarray:
    """Generate a non-neural alignment baseline using GR correlation and shifted TVT interpolation."""
    h_gr = pd.to_numeric(horizontal_df.get("GR"), errors="coerce").to_numpy(dtype=float)
    v_gr = pd.to_numeric(typewell_df.get("GR"), errors="coerce").to_numpy(dtype=float)
    v_tvt = pd.to_numeric(typewell_df.get("TVT"), errors="coerce").to_numpy(dtype=float)
    h_len = len(h_gr)
    v_len = len(v_tvt)
    lag = normalized_cross_correlation_lag(h_gr, v_gr)
    v_positions = np.arange(v_len, dtype=float) + lag * (max(h_len, v_len) - 1) / max(1, min(max(h_len, v_len), 512) - 1)
    v_positions = np.clip(v_positions, 0.0, max(v_len - 1, 0))
    target_positions = np.linspace(0.0, max(v_len - 1, 0), h_len)
    aligned = np.interp(target_positions, np.arange(v_len, dtype=float), _fill_series_for_alignment(v_tvt))
    return aligned.astype(np.float32)


classical_oof = np.zeros(len(train_tabular_df), dtype=np.float32)
fold_scores = []
unique_train_wells = sorted(train_well_map)
gkf = GroupKFold(n_splits=N_SPLITS)
well_groups = np.array(unique_train_wells)
for fold_idx, (_, val_well_idx) in enumerate(gkf.split(well_groups, well_groups, well_groups)):
    fold_wells = well_groups[val_well_idx]
    fold_rows = []
    fold_true = []
    fold_pred = []
    for well_name in fold_wells:
        bundle = train_well_map[well_name]
        pred = classical_alignment_predict(bundle["horizontal"], bundle["typewell"])
        mask = train_tabular_df["WELLNAME"].eq(well_name).to_numpy()
        classical_oof[mask] = pred[: mask.sum()]
        fold_rows.append(mask.sum())
        fold_true.append(train_tabular_df.loc[mask, "target_tvt"].to_numpy())
        fold_pred.append(pred[: mask.sum()])
    fold_true_arr = np.concatenate(fold_true)
    fold_pred_arr = np.concatenate(fold_pred)
    fold_rmse = rmse(fold_true_arr, fold_pred_arr)
    fold_scores.append(fold_rmse)
    print(f"Classical fold {fold_idx + 1}/{N_SPLITS} RMSE: {fold_rmse:.6f} | wells: {len(fold_wells)} | rows: {int(np.sum(fold_rows))}")

classical_rmse = rmse(y_train, classical_oof)
print(f"Classical alignment baseline OOF RMSE: {classical_rmse:.6f}")
print(f"Classical OOF shape: {classical_oof.shape}")

In [ ]:
WINDOW_HISTORY = 256
WINDOW_HORIZON = 1024
WINDOW_STRIDE = 512
SEQUENCE_FEATURE_DIM = 3


def _safe_fill(values: np.ndarray) -> np.ndarray:
    series = pd.Series(np.asarray(values, dtype=np.float32))
    series = series.interpolate(limit_direction="both").bfill().ffill()
    return series.to_numpy(dtype=np.float32)


def _zscore(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    values = _safe_fill(values)
    mean = float(np.nanmean(values))
    std = float(np.nanstd(values))
    std = std if std > 1e-6 else 1.0
    return (values - mean) / std


def _resample_1d(values: np.ndarray, target_len: int) -> np.ndarray:
    values = _safe_fill(values)
    if len(values) == target_len:
        return values.astype(np.float32)
    if len(values) == 0:
        return np.zeros(target_len, dtype=np.float32)
    x_src = np.linspace(0.0, 1.0, len(values))
    x_tgt = np.linspace(0.0, 1.0, target_len)
    return np.interp(x_tgt, x_src, values).astype(np.float32)


def build_window_spans(n_rows: int, start_idx: int, history: int, horizon: int, stride: int) -> List[Tuple[int, int, int, int]]:
    """Create overlapping sliding windows with a fixed history and bounded horizon."""
    spans = []
    if n_rows <= 0:
        return spans
    anchor = max(history, start_idx)
    if anchor >= n_rows:
        anchor = max(history, n_rows - 1)
    pos = anchor
    while pos < n_rows:
        input_start = max(0, pos - history)
        input_end = min(n_rows, pos + horizon)
        spans.append((input_start, pos, input_end, pos))
        if input_end >= n_rows:
            break
        pos += stride
    if not spans:
        spans.append((0, 0, min(n_rows, history + horizon), 0))
    return spans


class WindowSequenceDataset(torch.utils.data.Dataset):
    """Sliding-window dataset that constrains the sequence model to local chunks."""

    def __init__(self, wells: List[dict], include_target: bool = True, history: int = WINDOW_HISTORY, horizon: int = WINDOW_HORIZON, stride: int = WINDOW_STRIDE, mode: str = "train"):
        self.wells = wells
        self.include_target = include_target
        self.history = history
        self.horizon = horizon
        self.stride = stride
        self.mode = mode
        self.samples = []
        for bundle in wells:
            h = bundle["horizontal"].sort_values("row_idx").reset_index(drop=True)
            boundary = bundle["boundary"]
            start_idx = int(boundary["tvt_eval_start_idx"]) if mode in {"infer", "predict"} else history
            spans = build_window_spans(len(h), start_idx, history, horizon, stride)
            for span in spans:
                self.samples.append({"bundle": bundle, "span": span})

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        bundle = self.samples[idx]["bundle"]
        input_start, target_start, input_end, _ = self.samples[idx]["span"]
        h = bundle["horizontal"].sort_values("row_idx").reset_index(drop=True)
        v = bundle["typewell"].sort_values("row_idx").reset_index(drop=True)
        window = h.iloc[input_start:input_end].copy().reset_index(drop=True)
        window_len = len(window)
        if window_len == 0:
            window_len = 1
            window = h.iloc[:1].copy().reset_index(drop=True)
        h_gr = pd.to_numeric(window.get("GR"), errors="coerce").to_numpy(dtype=np.float32)
        h_md = pd.to_numeric(window.get("MD"), errors="coerce").to_numpy(dtype=np.float32)
        h_z = pd.to_numeric(window.get("Z"), errors="coerce").to_numpy(dtype=np.float32)
        h_feat = np.stack([
            _zscore(h_gr),
            _zscore(h_md),
            _zscore(h_z),
        ], axis=-1)

        v_gr = pd.to_numeric(v.get("GR"), errors="coerce").to_numpy(dtype=np.float32)
        v_resampled = _resample_1d(v_gr, window_len)
        v_pos = np.linspace(0.0, 1.0, window_len, dtype=np.float32)
        v_feat = np.stack([_zscore(v_resampled), v_pos], axis=-1)

        if "dtvt" in window.columns:
            target = pd.to_numeric(window["dtvt"], errors="coerce").to_numpy(dtype=np.float32)
        else:
            target = np.zeros(window_len, dtype=np.float32)
        target_mask = np.zeros(window_len, dtype=bool)
        target_mask[max(0, target_start - input_start):] = True
        target_mask = target_mask[:window_len]

        return {
            "wellname": bundle["WELLNAME"],
            "horizontal_features": torch.tensor(h_feat, dtype=torch.float32),
            "vertical_features": torch.tensor(v_feat, dtype=torch.float32),
            "target": torch.tensor(target, dtype=torch.float32),
            "target_mask": torch.tensor(target_mask, dtype=torch.bool),
            "row_idx": torch.tensor(window["row_idx"].to_numpy(dtype=np.int64), dtype=torch.long),
            "anchor_value": torch.tensor(float(bundle["boundary"]["tvt_last_valid_value"]), dtype=torch.float32),
        }


def sequence_collate(batch: List[dict]) -> dict:
    h_feats = [item["horizontal_features"] for item in batch]
    v_feats = [item["vertical_features"] for item in batch]
    targets = [item["target"] for item in batch]
    masks = [item["target_mask"] for item in batch]
    row_idx = [item["row_idx"] for item in batch]
    wellnames = [item["wellname"] for item in batch]
    anchors = torch.stack([item["anchor_value"] for item in batch], dim=0)
    return {
        "horizontal_features": torch.nn.utils.rnn.pad_sequence(h_feats, batch_first=True, padding_value=0.0),
        "vertical_features": torch.nn.utils.rnn.pad_sequence(v_feats, batch_first=True, padding_value=0.0),
        "target": torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=float("nan")),
        "target_mask": torch.nn.utils.rnn.pad_sequence(masks, batch_first=True, padding_value=0),
        "row_idx": row_idx,
        "wellname": wellnames,
        "anchor_value": anchors,
    }


class ConvEncoder(nn.Module):
    """Dual 1D CNN encoder for local stratigraphic signatures."""

    def __init__(self, input_dim: int = 3, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(input_dim, hidden_dim)
        self.block = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.out_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)
        x = x.transpose(1, 2)
        x = self.block(x)
        x = x.transpose(1, 2)
        return self.out_norm(x)


class CrossAttentionBlock(nn.Module):
    """Cross-attention with dtype-safe mask fill for mixed precision."""

    def __init__(self, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = float(hidden_dim) ** -0.5

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, q_mask: torch.Tensor | None = None, kv_mask: torch.Tensor | None = None) -> Tuple[torch.Tensor, torch.Tensor]:
        q = self.q_proj(q)
        k = self.k_proj(k)
        v = self.v_proj(v)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if kv_mask is not None:
            min_val = torch.finfo(scores.dtype).min
            scores = scores.masked_fill(~kv_mask.unsqueeze(1), min_val)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context = torch.matmul(attn, v)
        return self.out_proj(context), attn


class WindowAlignmentNet(nn.Module):
    """Windowed alignment network that predicts dtvt within a bounded chunk."""

    def __init__(self, input_dim_h: int = 3, input_dim_v: int = 2, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.horizontal_encoder = ConvEncoder(input_dim=input_dim_h, hidden_dim=hidden_dim, dropout=dropout)
        self.vertical_encoder = ConvEncoder(input_dim=input_dim_v, hidden_dim=hidden_dim, dropout=dropout)
        self.cross_attention = CrossAttentionBlock(hidden_dim=hidden_dim, dropout=dropout)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, horizontal_features: torch.Tensor, vertical_features: torch.Tensor, horizontal_mask: torch.Tensor | None = None, vertical_mask: torch.Tensor | None = None) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.horizontal_encoder(horizontal_features)
        v = self.vertical_encoder(vertical_features)
        context, attn = self.cross_attention(h, v, v, q_mask=horizontal_mask, kv_mask=vertical_mask)
        fusion = torch.cat([h, context, h - context, h * context], dim=-1)
        pred = self.regressor(fusion).squeeze(-1)
        return pred, attn


def masked_dtvt_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    valid = mask & torch.isfinite(target)
    if not valid.any():
        return pred.sum() * 0.0
    pred_valid = pred[valid]
    target_valid = target[valid]
    mse = F.mse_loss(pred_valid, target_valid)
    huber = F.smooth_l1_loss(pred_valid, target_valid, beta=1.0)
    return 0.5 * mse + 0.5 * huber


def monotonicity_penalty(pred: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    if pred.size(1) < 3:
        return pred.sum() * 0.0
    abs_pred = torch.cumsum(pred, dim=1)
    d1 = abs_pred[:, 1:] - abs_pred[:, :-1]
    d2 = d1[:, 1:] - d1[:, :-1]
    valid1 = mask[:, 1:] & mask[:, :-1]
    valid2 = mask[:, 2:] & mask[:, 1:-1] & mask[:, :-2]
    p1 = (d1.pow(2) * valid1).sum() / valid1.sum().clamp_min(1)
    p2 = (d2.pow(2) * valid2).sum() / valid2.sum().clamp_min(1)
    return 0.25 * p1 + 0.75 * p2


def predict_sequence_windows(model: nn.Module, dataset: WindowSequenceDataset, batch_size: int = 2) -> dict:
    """Average overlapping window predictions back to per-row well sequences."""
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=sequence_collate)
    per_well = {}
    for batch in loader:
        horizontal = batch["horizontal_features"].to(DEVICE)
        vertical = batch["vertical_features"].to(DEVICE)
        h_mask = torch.ones(horizontal.shape[:2], device=DEVICE, dtype=torch.bool)
        v_mask = torch.ones(vertical.shape[:2], device=DEVICE, dtype=torch.bool)
        with torch.no_grad():
            pred, _ = model(horizontal, vertical, h_mask, v_mask)
        pred = pred.detach().cpu().numpy()
        row_idx_list = batch["row_idx"]
        wellnames = batch["wellname"]
        for i, wellname in enumerate(wellnames):
            row_idx = np.asarray(row_idx_list[i], dtype=np.int64)
            row_pred = pred[i, : len(row_idx)]
            if wellname not in per_well:
                per_well[wellname] = {"sum": np.zeros(int(row_idx.max()) + 1, dtype=np.float32), "count": np.zeros(int(row_idx.max()) + 1, dtype=np.float32)}
            if int(row_idx.max()) >= len(per_well[wellname]["sum"]):
                grow = int(row_idx.max()) + 1 - len(per_well[wellname]["sum"])
                per_well[wellname]["sum"] = np.pad(per_well[wellname]["sum"], (0, grow))
                per_well[wellname]["count"] = np.pad(per_well[wellname]["count"], (0, grow))
            np.add.at(per_well[wellname]["sum"], row_idx, row_pred)
            np.add.at(per_well[wellname]["count"], row_idx, 1.0)
    averaged = {}
    for wellname, buffers in per_well.items():
        counts = buffers["count"]
        sums = buffers["sum"]
        preds = np.divide(sums, np.maximum(counts, 1.0), out=np.zeros_like(sums), where=counts > 0)
        averaged[wellname] = preds.astype(np.float32)
    return averaged


def _sequence_rmse_from_predictions(wells: List[dict], predictions: dict) -> float:
    actual = []
    pred = []
    for bundle in wells:
        h = bundle["horizontal"].sort_values("row_idx").reset_index(drop=True)
        y_true = pd.to_numeric(h["dtvt"], errors="coerce").to_numpy(dtype=np.float32)
        y_pred = np.asarray(predictions[bundle["WELLNAME"]], dtype=np.float32)
        if len(y_pred) < len(y_true):
            y_pred = np.pad(y_pred, (0, len(y_true) - len(y_pred)), constant_values=0.0)
        actual.append(y_true)
        pred.append(y_pred[: len(y_true)])
    actual = np.concatenate(actual)
    pred = np.concatenate(pred)
    return rmse(actual, pred)


def train_sequence_fold(train_wells_fold: List[dict], val_wells_fold: List[dict], epochs: int = MAX_SEQ_EPOCHS, batch_size: int = 2) -> Tuple[nn.Module, float, dict]:
    train_dataset = WindowSequenceDataset(train_wells_fold, include_target=True, history=WINDOW_HISTORY, horizon=WINDOW_HORIZON, stride=WINDOW_STRIDE, mode="train")
    val_dataset = WindowSequenceDataset(val_wells_fold, include_target=True, history=WINDOW_HISTORY, horizon=WINDOW_HORIZON, stride=WINDOW_STRIDE, mode="infer")
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=sequence_collate)
    sample_batch = next(iter(train_loader))
    print(f"Sequence window batch shapes -> H: {tuple(sample_batch['horizontal_features'].shape)}, V: {tuple(sample_batch['vertical_features'].shape)}, y: {tuple(sample_batch['target'].shape)}")

    model = WindowAlignmentNet(input_dim_h=3, input_dim_v=2, hidden_dim=64, dropout=0.1).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_CUDA)

    best_state = None
    best_rmse = float("inf")
    stale_epochs = 0
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for batch in train_loader:
            horizontal = batch["horizontal_features"].to(DEVICE)
            vertical = batch["vertical_features"].to(DEVICE)
            target = batch["target"].to(DEVICE)
            mask = batch["target_mask"].to(DEVICE).bool()
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_CUDA):
                pred, attn = model(horizontal, vertical)
                base_loss = masked_dtvt_loss(pred, target, mask)
                smooth_penalty = monotonicity_penalty(pred, mask)
                loss = base_loss + 0.1 * smooth_penalty
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.detach().cpu().item()))

        model.eval()
        val_predictions = predict_sequence_windows(model, val_dataset, batch_size=batch_size)
        epoch_rmse = _sequence_rmse_from_predictions(val_wells_fold, val_predictions)
        scheduler.step(epoch_rmse)
        print(f"Sequence epoch {epoch + 1:02d}/{epochs} | train loss {np.mean(train_losses):.6f} | val dtvt RMSE {epoch_rmse:.6f}")
        if epoch_rmse < best_rmse - 1e-6:
            best_rmse = epoch_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= SEQ_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_val_predictions = predict_sequence_windows(model, val_dataset, batch_size=batch_size)
    return model, best_rmse, final_val_predictions


sequence_oof = np.full(len(train_tabular_df), np.nan, dtype=np.float32)
sequence_test_predictions_by_well = {}
sequence_fold_scores = []
sequence_models = []
train_well_names_order = np.array(sorted(train_well_map))
well_group_kfold = GroupKFold(n_splits=N_SPLITS)
for fold_idx, (train_well_idx, val_well_idx) in enumerate(well_group_kfold.split(train_well_names_order, train_well_names_order, train_well_names_order)):
    train_fold_names = train_well_names_order[train_well_idx].tolist()
    val_fold_names = train_well_names_order[val_well_idx].tolist()
    train_fold_wells = [train_well_map[name] for name in train_fold_names]
    val_fold_wells = [train_well_map[name] for name in val_fold_names]
    print(f"\nSequence fold {fold_idx + 1}/{N_SPLITS} | train wells: {len(train_fold_wells)} | val wells: {len(val_fold_wells)}")
    model, fold_rmse, val_predictions = train_sequence_fold(train_fold_wells, val_fold_wells, epochs=MAX_SEQ_EPOCHS, batch_size=2)
    sequence_models.append(model)
    sequence_fold_scores.append(fold_rmse)
    for well_name, pred in val_predictions.items():
        mask = train_tabular_df["WELLNAME"].eq(well_name).to_numpy()
        y = np.asarray(pred, dtype=np.float32)
        if len(y) < mask.sum():
            y = np.pad(y, (0, mask.sum() - len(y)), constant_values=0.0)
        sequence_oof[mask] = y[: mask.sum()]
    test_dataset = WindowSequenceDataset(test_wells, include_target=False, history=WINDOW_HISTORY, horizon=WINDOW_HORIZON, stride=WINDOW_STRIDE, mode="infer")
    fold_test_predictions = predict_sequence_windows(model, test_dataset, batch_size=2)
    for well_name, pred in fold_test_predictions.items():
        if well_name not in sequence_test_predictions_by_well:
            sequence_test_predictions_by_well[well_name] = []
        sequence_test_predictions_by_well[well_name].append(pred)
    print(f"Sequence fold {fold_idx + 1} validation dtvt RMSE: {fold_rmse:.6f}")

for well_name, preds in sequence_test_predictions_by_well.items():
    stacked = np.vstack([np.asarray(p, dtype=np.float32) for p in preds])
    sequence_test_predictions_by_well[well_name] = np.nanmean(stacked, axis=0).astype(np.float32)

sequence_oof = np.nan_to_num(sequence_oof, nan=0.0)
sequence_rmse = rmse(y_train, sequence_oof)
print(f"Sequence dtvt OOF RMSE: {sequence_rmse:.6f}")
print(f"Sequence OOF shape: {sequence_oof.shape}")
print(f"Sequence test predictions prepared for {len(sequence_test_predictions_by_well)} wells")


In [ ]:
def _stack_oof_predictions(*arrays: np.ndarray) -> np.ndarray:
    return np.column_stack([np.asarray(a, dtype=np.float32) for a in arrays])


def fit_oof_ridge(meta_features: np.ndarray, y: np.ndarray, groups: np.ndarray, n_splits: int = N_SPLITS, alpha: float = 1.0):
    """Train an OOF Ridge meta-learner with GroupKFold on dtvt targets."""
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(y), dtype=np.float32)
    fold_models = []
    fold_rmses = []
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(meta_features, y, groups)):
        model = Ridge(alpha=alpha, random_state=SEED)
        model.fit(meta_features[train_idx], y[train_idx])
        val_pred = model.predict(meta_features[val_idx])
        oof[val_idx] = val_pred
        fold_models.append(model)
        fold_rmses.append(rmse(y[val_idx], val_pred))
        print(f"Ridge meta fold {fold_idx + 1}/{n_splits} dtvt RMSE: {fold_rmses[-1]:.6f}")
    overall_rmse = rmse(y, oof)
    return oof, fold_models, overall_rmse


train_tabular_df["oof_sequence_dtvt"] = sequence_oof

meta_feature_columns_tabular = ["oof_lgbm_dtvt", "oof_catboost_dtvt", "oof_xgb_dtvt"]
meta_feature_columns_full = meta_feature_columns_tabular + ["oof_sequence_dtvt"]

tabular_oof_stack = train_tabular_df[meta_feature_columns_tabular].to_numpy(dtype=np.float32)
full_oof_stack = train_tabular_df[meta_feature_columns_full].to_numpy(dtype=np.float32)

tabular_meta_oof, tabular_meta_models, tabular_meta_rmse = fit_oof_ridge(tabular_oof_stack, y_train, groups, n_splits=N_SPLITS, alpha=1.0)
full_meta_oof, full_meta_models, full_meta_rmse = fit_oof_ridge(full_oof_stack, y_train, groups, n_splits=N_SPLITS, alpha=1.0)

sequence_gate_rmse = sequence_rmse
use_full_fusion = bool(full_meta_rmse < tabular_meta_rmse and sequence_gate_rmse <= tabular_meta_rmse * 1.05)
blend_mode = "full_fusion" if use_full_fusion else "tabular_only"
final_oof = full_meta_oof if use_full_fusion else tabular_meta_oof
final_meta_features_train = full_oof_stack if use_full_fusion else tabular_oof_stack
final_meta_model = Ridge(alpha=1.0, random_state=SEED).fit(final_meta_features_train, y_train)
final_train_rmse = rmse(y_train, final_oof)

print(f"Tabular-only Ridge OOF dtvt RMSE: {tabular_meta_rmse:.6f}")
print(f"Full fusion Ridge OOF dtvt RMSE: {full_meta_rmse:.6f}")
print(f"Sequence gate dtvt RMSE: {sequence_gate_rmse:.6f}")
print(f"Selected blend mode: {blend_mode}")
print(f"Final blended OOF dtvt RMSE: {final_train_rmse:.6f}")
print(f"Meta feature shapes -> tabular: {tabular_oof_stack.shape}, full: {full_oof_stack.shape}")


def compute_error_decomposition(df: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """Create per-well diagnostic metrics for residual analysis."""
    rows = []
    for well_name, well_df in df.groupby("WELLNAME", sort=False):
        idx = well_df.index.to_numpy()
        actual = y_true[idx]
        pred = y_pred[idx]
        err = pred - actual
        rows.append(
            {
                "WELLNAME": well_name,
                "n_rows": int(len(idx)),
                "rmse": rmse(actual, pred),
                "mae": float(mean_absolute_error(actual, pred)),
                "directional_bias": float(err.mean()),
                "spatial_drift": float(np.abs(np.cumsum(err)).mean()),
            }
        )
    return pd.DataFrame(rows).sort_values("rmse", ascending=False).reset_index(drop=True)


error_report_df = compute_error_decomposition(train_tabular_df, y_train, final_oof)
print(f"Error decomposition report shape: {error_report_df.shape}")
print(error_report_df.head())


def plot_well_alignment(wellname: str, md: np.ndarray, actual: np.ndarray, predicted: np.ndarray, max_points: int = 400) -> None:
    """Plot predicted vs actual dtvt along measured depth for one well."""
    plt.figure(figsize=(12, 4))
    if len(md) > max_points:
        stride = max(1, len(md) // max_points)
        md = md[::stride]
        actual = actual[::stride]
        predicted = predicted[::stride]
    plt.plot(md, actual, label="Actual dtvt", linewidth=2)
    plt.plot(md, predicted, label="Predicted dtvt", linewidth=2)
    plt.title(f"{wellname} - Predicted dtvt vs Actual dtvt")
    plt.xlabel("Measured Depth (MD)")
    plt.ylabel("dtvt")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


example_well = error_report_df.iloc[0]["WELLNAME"] if len(error_report_df) else train_tabular_df["WELLNAME"].iloc[0]
example_mask = train_tabular_df["WELLNAME"].eq(example_well).to_numpy()
example_md = pd.to_numeric(train_tabular_df.loc[example_mask, "MD"], errors="coerce").to_numpy(dtype=float)
example_actual = y_train[example_mask]
example_pred = final_oof[example_mask]
print(f"Plot diagnostic example: {example_well} | rows: {example_mask.sum()}")
plot_well_alignment(example_well, example_md, example_actual, example_pred)


In [ ]:
def _average_fold_test_predictions(per_fold_predictions: List[Dict[str, np.ndarray]], well_names: List[str]) -> Dict[str, np.ndarray]:
    """Average per-fold well predictions into a single test prediction per well."""
    well_to_predictions = {name: [] for name in well_names}
    for fold_pred in per_fold_predictions:
        for name in well_names:
            if name in fold_pred:
                well_to_predictions[name].append(fold_pred[name])
    averaged = {}
    for name, preds in well_to_predictions.items():
        if preds:
            stacked = np.vstack([np.asarray(p, dtype=np.float32) for p in preds])
            averaged[name] = np.nanmean(stacked, axis=0).astype(np.float32)
        else:
            averaged[name] = np.full(len(test_well_map[name]["horizontal"]), np.nan, dtype=np.float32)
    return averaged


# Base test predictions for tabular models come from fold-averaged predictions already computed in cell 3.
tabular_test_stack = _stack_oof_predictions(
    tabular_results["lgbm"]["test_pred"],
    tabular_results["catboost"]["test_pred"],
    tabular_results["xgboost"]["test_pred"],
)

# Sequence predictions are now row-wise dtvt values aggregated from sliding windows.
sequence_test_dtvt_by_well = sequence_test_predictions_by_well
print(f"Sequence test dtvt coverage: {len(sequence_test_dtvt_by_well)} wells")

sequence_test_vector = np.concatenate([sequence_test_dtvt_by_well[name] for name in test_well_map.keys()]).astype(np.float32)

if use_full_fusion:
    test_meta_features = _stack_oof_predictions(
        tabular_results["lgbm"]["test_pred"],
        tabular_results["catboost"]["test_pred"],
        tabular_results["xgboost"]["test_pred"],
        sequence_test_vector,
    )
else:
    test_meta_features = tabular_test_stack

final_test_dtvt_all_rows = final_meta_model.predict(test_meta_features).astype(np.float32)
final_test_dtvt_all_rows = np.nan_to_num(final_test_dtvt_all_rows, nan=0.0, posinf=0.0, neginf=0.0)

test_tabular_df = test_tabular_df.sort_values(["WELLNAME", "row_idx"]).reset_index(drop=True).copy()
test_tabular_df["final_dtvt_prediction"] = final_test_dtvt_all_rows

submission_rows = []
for well_name, bundle in test_well_map.items():
    horizontal = bundle["horizontal"].sort_values("row_idx").reset_index(drop=True)
    boundary = bundle["boundary"]
    eval_start_idx = int(boundary["tvt_eval_start_idx"])
    anchor_value = float(boundary["tvt_last_valid_value"])
    mask = horizontal["row_idx"].to_numpy() >= eval_start_idx
    well_dtvt = test_tabular_df.loc[test_tabular_df["WELLNAME"].eq(well_name), "final_dtvt_prediction"].to_numpy(dtype=np.float32)
    if len(well_dtvt) < len(horizontal):
        well_dtvt = np.pad(well_dtvt, (0, len(horizontal) - len(well_dtvt)), constant_values=0.0)
    hidden_dtvt = well_dtvt[mask]
    well_tvt = anchor_value + np.cumsum(hidden_dtvt)
    well_rows = horizontal.loc[mask, ["WELLNAME", "row_idx"]].copy()
    well_rows["id"] = well_rows["WELLNAME"].astype(str) + "_" + well_rows["row_idx"].astype(int).astype(str)
    well_rows["tvt"] = well_tvt.astype(np.float32)
    submission_rows.append(well_rows[["id", "tvt"]])

submission_df = pd.concat(submission_rows, ignore_index=True)
submission_df = submission_df.dropna(subset=["id", "tvt"]).copy()
submission_df["tvt"] = submission_df["tvt"].astype(float)
assert not submission_df["tvt"].isna().any(), "Submission contains NaN TVT values."
submission_df.to_csv("submission.csv", index=False)

print(f"Submission shape: {submission_df.shape}")
print(submission_df.head())
print(f"Final blend mode used for test predictions: {blend_mode}")
print("Submission saved to: submission.csv")
